# Hypothetical Document Embedding (HyDE)

## Overview

Standard RAG retrieval matches a **short query** against **long document chunks** — but these live in very different parts of the embedding space. HyDE bridges this gap:

| Standard Retrieval | HyDE Retrieval |
|---|---|
| Query → embed → search | Query → **generate hypothetical answer** → embed answer → search |
| Short query vs. long chunks (mismatch) | Full document vs. long chunks (better match) |

The idea: ask the LLM to *imagine* what a document answering the question would look like, then search for real documents similar to that imagined one.

## Models Used

- **LLM**: `gemma3:4b` via Ollama (local) — generates the hypothetical document
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

<div style="text-align: center;">

<img src="./images/HyDe.svg" alt="HyDe" style="width:40%; height:auto;">
</div>

<div style="text-align: center;">

<img src="./images/hyde-advantages.svg" alt="HyDe" style="width:100%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
from langchain_ollama import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

---
## Step 1: Set Up LLM and Embedding Model

In [2]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")
llm = ChatOllama(model="gemma3:4b", temperature=0, max_tokens=4000)

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Load PDF, Chunk, and Build the Vector Store

In [3]:
path = "data/Understanding_Climate_Change.pdf"
chunk_size = 500
chunk_overlap = 100

loader = PyPDFLoader(path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
)
texts = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(texts, embedding_model)

print(f"Loaded {len(documents)} pages, split into {len(texts)} chunks (size={chunk_size}, overlap={chunk_overlap})")
print("Vector store created")

Loaded 33 pages, split into 201 chunks (size=500, overlap=100)
Vector store created


---
## Step 3: Define the HyDE Prompt

This prompt asks the LLM to **imagine a document** that directly answers the question. The generated text should be roughly the same length as our chunk size, so its embedding lands in the same neighborhood as real chunks.

In [4]:
hyde_prompt = PromptTemplate(
    input_variables=["query", "chunk_size"],
    template=(
        "Given the question '{query}', generate a hypothetical document that directly answers this question. "
        "The document should be detailed and in-depth. "
        "The document size has to be exactly {chunk_size} characters."
    )
)

hyde_chain = hyde_prompt | llm

print("HyDE prompt chain ready")

HyDE prompt chain ready


---
## Step 4: Generate the Hypothetical Document

Given our test query, the LLM generates a fake (but plausible) answer document. This document doesn't need to be factually perfect — it just needs to be *semantically similar* to real documents that contain the answer.

In [5]:
test_query = "What is the main cause of climate change?"
print(f"Query: {test_query}\n")

hypothetical_doc = hyde_chain.invoke({"query": test_query, "chunk_size": chunk_size}).content

print("Generated hypothetical document:")
print("-" * 60)
print(hypothetical_doc)
print("-" * 60)
print(f"Length: {len(hypothetical_doc)} characters")

Query: What is the main cause of climate change?

Generated hypothetical document:
------------------------------------------------------------
Okay, here’s a 500-character document answering “What is the main cause of climate change?”

---

**Report: Climate Change – Primary Driver**

The overwhelming scientific consensus identifies **anthropogenic greenhouse gas emissions** as the primary driver of current climate change. Specifically, the burning of fossil fuels (coal, oil, and natural gas) releases vast quantities of carbon dioxide (CO2) into the atmosphere. Deforestation further exacerbates the issue by reducing the planet’s capacity to absorb CO2. Methane and nitrous oxide, also released by human activities, contribute significantly.  Rising global temperatures are now undeniably linked to this increased atmospheric concentration.  Continued emissions will intensify the effects.

---

(Character count: 488)

------------------------------------------------------------
Length: 783

---
## Step 5: Retrieve Real Documents Using the Hypothetical Document

Instead of searching with the short query, we search with the **hypothetical document** as the query. This produces a much better embedding match with real chunks.

In [6]:
k = 3
results = vectorstore.similarity_search(hypothetical_doc, k=k)

print(f"Retrieved {len(results)} documents:\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(doc.page_content)
    print()

Retrieved 3 documents:

--- Result 1 ---
predict future trends. The evidence overwhelmingly shows that recent changes are primarily 
driven by human activities, particularly the emission of greenhouse gases. 
Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." This effect is essential

--- Result 2 ---
Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an extended period. Over the past century, human 
activities, particularly the burning of fossil fuels and deforestation, have significantly 
contributed to

---
## Step 6 (Optional): Compare HyDE vs. Standard Retrieval

Let's see what we'd get if we searched with the **original short query** instead of the hypothetical document.

In [7]:
results_standard = vectorstore.similarity_search(test_query, k=k)

print("=== Standard Retrieval (short query) ===")
for i, doc in enumerate(results_standard, 1):
    print(f"  {i}) {doc.page_content[:150]}...")

print("\n=== HyDE Retrieval (hypothetical document) ===")
for i, doc in enumerate(results, 1):
    print(f"  {i}) {doc.page_content[:150]}...")

print("\nNotice how HyDE may retrieve more relevant or differently-ranked chunks,")
print("because the hypothetical document is semantically closer to the real answer chunks.")

=== Standard Retrieval (short query) ===
  1) predict future trends. The evidence overwhelmingly shows that recent changes are primarily 
driven by human activities, particularly the emission of g...
  2) Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate...
  3) and infrastructure. Cities are particularly vulnerable due to the "urban heat island" effect. 
Heatwaves can lead to heat-related illnesses and exacer...

=== HyDE Retrieval (hypothetical document) ===
  1) predict future trends. The evidence overwhelmingly shows that recent changes are primarily 
driven by human activities, particularly the emission of g...
  2) Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate...
  3) Modern scientific observations indicate a rapid increase in global temperatures, sea levels, 
and extreme weath

---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up local LLM + embeddings |
| 2 | Loaded PDF, chunked, built FAISS vector store |
| 3 | Defined HyDE prompt ("imagine a document that answers this") |
| 4 | Generated a hypothetical answer document from the query |
| 5 | Used the hypothetical document (not the query) to search the vector store |
| 6 | Compared HyDE vs. standard retrieval |

**Key insight:** HyDE works because embedding a *document-length* text produces a vector in the same region of the embedding space as real document chunks. A short query like "What is the main cause of climate change?" lives far from the chunk embeddings, but a generated paragraph about greenhouse gases and fossil fuels lands right next to the real answer chunks.